In [1]:
"""
Smote only layer 2 model training

Author: Tan Yi Feng
Student ID: 23WMR14766
"""
import pandas as pd
import numpy as np
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import joblib
import pickle
import pandas as pd


SCRIPT_DIR = Path.cwd()


DATA_PATH = SCRIPT_DIR / 'data' / 'processed' / 'FYP_Proccesed_Data'/'balanced_train_data.csv'
CONFIG_PATH = SCRIPT_DIR / 'config' / 'config.yaml'
MODELS_DIR = SCRIPT_DIR / 'data' / 'models'
SRC_PATH = SCRIPT_DIR / 'src'


print("Loading data for SMOTE-Only Model...")
df = pd.read_csv(DATA_PATH)
df = df[df['label'] != 0].copy() 


label_merge_map = {
    1: 1, 2: 2, 3: 3, 4: 2, 5: 3, 
    6: 4, 7: 5, 8: 6, 9: 5, 10: 5, 
    11: 7, 15: 7, 12: 8, 16: 8, 
    13: 9, 17: 9, 14: 10, 18: 10
}
df['label'] = df['label'].map(label_merge_map)


df.replace([np.inf, -np.inf], np.nan, inplace=True)
X = df.drop(columns=['label']).select_dtypes(include=['number'])
y = df['label']

X = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(X), columns=X.columns)
y = LabelEncoder().fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


classes = np.unique(y_train)
base_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, base_weights))

penalty_classes = [1, 2, 3, 4, 5, 6] 
for cls in penalty_classes:
    if cls in class_weight_dict:
        class_weight_dict[cls] *= 2.0


print(f"\nTraining SMOTE-Only Model with {X_train.shape[1]} features...")

smote_pipeline = Pipeline([
    ('smote', SMOTE(sampling_strategy='auto', k_neighbors=5, random_state=42)),
    ('rf', RandomForestClassifier(
        random_state=42,
        n_jobs=-1,
        class_weight=class_weight_dict
    ))
])

t0 = time.time()
smote_pipeline.fit(X_train, y_train)
train_time = time.time() - t0


pred_start = time.time()
y_pred = smote_pipeline.predict(X_test)
inference_time = time.time() - pred_start

acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average='macro')
f1_weighted = f1_score(y_test, y_pred, average='weighted')

print("\n=== SMOTE-ONLY PERFORMANCE (NO GA) ===")
print(f"Features Used   : {X_train.shape[1]} (All features)")
print(f"Training Time   : {train_time:.2f} s")
print(f"Inference Time  : {inference_time:.4f} s")
print(f"Avg per Packet  : {(inference_time/len(X_test)):.6f} s")
print(f"Accuracy        : {acc:.4f}")
print(f"Macro F1        : {f1_macro:.4f}")
print(f"Weighted F1     : {f1_weighted:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Loading data for SMOTE-Only Model...

Training SMOTE-Only Model with 39 features...

=== SMOTE-ONLY PERFORMANCE (NO GA) ===
Features Used   : 39 (All features)
Training Time   : 2.60 s
Inference Time  : 0.2308 s
Avg per Packet  : 0.000058 s
Accuracy        : 0.9717
Macro F1        : 0.9371
Weighted F1     : 0.9721

Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.73      0.71       160
           1       1.00      0.99      1.00       320
           2       1.00      1.00      1.00       320
           3       0.82      0.89      0.85       160
           4       0.97      0.93      0.95       480
           5       0.86      0.86      0.86       112
           6       1.00      1.00      1.00       720
           7       1.00      1.00      1.00       480
           8       1.00      1.00      1.00       480
           9       1.00      1.00      1.00       720

    accuracy                           0.97      3952
   macro a

In [3]:
import joblib
import os


model_filename = 'smote_rf_model_v1.pkl'


print(f"\nSaving model to {model_filename}...")
joblib.dump(smote_pipeline, model_filename)

print("Model saved successfully!")


feature_names = X_train.columns.tolist()
joblib.dump(feature_names, 'SmoteOnly.pkl')


Saving model to smote_rf_model_v1.pkl...
Model saved successfully!


['SmoteOnly.pkl']